# SIEWS+ — [GANTI: Nama Stage] Training Template
## YOLO26n Transfer Learning

**CARA PAKAI TEMPLATE INI:**
1. Ganti semua `[GANTI: ...]` sesuai dataset baru
2. Jalankan Cell 1–5 dahulu untuk verifikasi kelas
3. Isi `CLASS_NAMES` di Cell 5 berdasarkan hasil visualisasi
4. Re-run Cell 6 (visualisasi ulang dengan nama) untuk konfirmasi
5. Baru lanjut ke Cell 7 (Training)

**Dataset:** [GANTI: nama dataset]
**ZIP file:** [GANTI: nama file .zip]

---
**Kaggle:** Upload ZIP → Add Data → GPU T4 → Run All → Download `.pt`

## 1. Setup

In [ ]:
import os, sys, subprocess, zipfile, shutil, random, yaml, traceback
from pathlib import Path
from collections import Counter

IS_KAGGLE = os.path.exists('/kaggle/input')
print(f'Kaggle: {IS_KAGGLE}')

subprocess.run([sys.executable,'-m','pip','install','-U','ultralytics','-q'], check=True)
import ultralytics, torch
print(f'Ultralytics : {ultralytics.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
DEVICE = '0' if torch.cuda.is_available() else 'cpu'

## 2. Path Config
**Ganti `ZIP_NAME` dan `RUN_NAME` sesuai dataset.**

In [ ]:
# ============================================================
# GANTI DI SINI
# ============================================================
ZIP_NAME = '[GANTI: nama file .zip]'   # contoh: 'no safety harness.v2i.yolo26.zip'
RUN_NAME = '[GANTI: nama run]'         # contoh: 'ppe_harness'
OUT_NAME = '[GANTI: nama output .pt]'  # contoh: 'best_stage2_ppe.pt'
# ============================================================

if IS_KAGGLE:
    DATASET_ZIP = None
    OUTPUT_DIR  = Path('/kaggle/working')
    EXTRACT_DIR = None
else:
    DATASET_ZIP = Path('F:/migas-siews/dataset') / ZIP_NAME
    OUTPUT_DIR  = Path(f'F:/migas-siews/training/runs_{RUN_NAME}')
    EXTRACT_DIR = OUTPUT_DIR / 'dataset'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_DIR / 'runs' / RUN_NAME
if DATASET_ZIP:
    print(f'ZIP    : {DATASET_ZIP}')
    print(f'Exists : {DATASET_ZIP.exists()}')
else:
    print('Kaggle mode')

## 3. Ekstrak Dataset

In [ ]:
if IS_KAGGLE:
    kaggle_input = Path('/kaggle/input')
    found = None
    for candidate in kaggle_input.rglob('train'):
        if (candidate / 'images').exists():
            found = candidate.parent
            break
    if found is None:
        subdirs = [d for d in kaggle_input.iterdir() if d.is_dir()]
        print('Isi /kaggle/input/:', [d.name for d in subdirs])
        found = subdirs[0] if subdirs else None
    if found is None:
        raise FileNotFoundError('Dataset tidak ditemukan! Pastikan sudah Add Data di Kaggle.')
    EXTRACT_DIR = found
    print(f'Dataset: {EXTRACT_DIR}')

elif DATASET_ZIP and DATASET_ZIP.exists():
    if not EXTRACT_DIR.exists():
        print('Extracting...')
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
        print('Done!')
    else:
        print('Already extracted')
else:
    raise FileNotFoundError(f'ZIP tidak ada: {DATASET_ZIP}')

print()
for split in ['train', 'valid', 'val', 'test']:
    p = EXTRACT_DIR / split / 'images'
    if p.exists():
        ni = len(list(p.glob('*.*')))
        nl_dir = EXTRACT_DIR / split / 'labels'
        nl = len(list(nl_dir.glob('*.txt'))) if nl_dir.exists() else 0
        print(f'[{split:6s}] images={ni:5d}  labels={nl:5d}')

## 4. Baca README & Distribusi Class
**Baca output ini dengan teliti sebelum mengisi CLASS_NAMES di Cell 5!**

In [ ]:
# Baca README dari ZIP (jika ada)
if DATASET_ZIP and DATASET_ZIP.exists():
    with zipfile.ZipFile(DATASET_ZIP) as zf:
        for fname in ['README.dataset.txt', 'README.roboflow.txt', 'README.txt', 'notes.json']:
            if fname in zf.namelist():
                print(f'=== {fname} ===')
                print(zf.read(fname).decode()[:800])
                print()

In [ ]:
# Hitung distribusi class dari semua label files
counts = Counter()
lbl_dir = EXTRACT_DIR / 'train' / 'labels'
if not lbl_dir.exists():
    lbl_dir = EXTRACT_DIR / 'valid' / 'labels'

for lf in lbl_dir.glob('*.txt'):
    for line in lf.read_text().strip().splitlines():
        if line.strip():
            try: counts[int(line.split()[0])] += 1
            except: pass

if not counts:
    print('TIDAK ADA LABEL! Cek struktur folder dataset.')
else:
    print('Class distribution (hanya angka — belum ada nama):')
    total_instances = sum(counts.values())
    for cid in sorted(counts):
        pct = counts[cid]/total_instances*100
        bar = '#' * (counts[cid]//20)
        print(f'  class {cid}: {counts[cid]:5d} instances ({pct:4.1f}%)  {bar}')
    print(f'\nJumlah kelas: {len(counts)}')
    print(f'Max class ID : {max(counts)}')
    print()
    print('PERHATIKAN: Catat jumlah kelas ini dan lihat visualisasi di Cell berikutnya')
    print('           SEBELUM mengisi CLASS_NAMES!')

## 5. Visualisasi dengan CLASS ID (Angka) — Cek Dulu!
**Lihat visualisasi ini, perhatikan gambar → lalu tentukan nama class yang benar.**

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

# Warna berbeda per class ID (tanpa nama dulu)
COLOR_PALETTE = [
    (255,  80,   0),  # ID 0 — orange
    (  0, 200, 255),  # ID 1 — cyan
    (255,   0,   0),  # ID 2 — red
    (255, 220,   0),  # ID 3 — yellow
    (  0, 255, 100),  # ID 4 — green
    (180,   0, 255),  # ID 5 — purple
    (  0, 100, 255),  # ID 6 — blue
    (255, 150, 150),  # ID 7 — pink
]

train_img_dir = EXTRACT_DIR / 'train' / 'images'
train_lbl_dir = EXTRACT_DIR / 'train' / 'labels'
all_imgs = list(train_img_dir.glob('*.jpg')) + list(train_img_dir.glob('*.png'))
if not all_imgs:
    raise FileNotFoundError(f'Tidak ada gambar di {train_img_dir}')

samples = random.sample(all_imgs, min(9, len(all_imgs)))
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for ax, ip in zip(axes.flat, samples):
    img = cv2.imread(str(ip))
    if img is None: ax.axis('off'); continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lp = train_lbl_dir / (ip.stem + '.txt')
    ids_in_img = []
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5: continue
            cid = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = int((xc-bw/2)*w); y1 = int((yc-bh/2)*h)
            x2 = int((xc+bw/2)*w); y2 = int((yc+bh/2)*h)
            col = COLOR_PALETTE[cid % len(COLOR_PALETTE)]
            cv2.rectangle(img, (x1,y1), (x2,y2), col, 3)
            # Tampilkan ID angka saja
            label = f'ID:{cid}'
            cv2.putText(img, label, (x1, max(y1-5,15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, col, 2)
            ids_in_img.append(cid)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'IDs: {sorted(set(ids_in_img))}', fontsize=10, fontweight='bold')

plt.suptitle('STEP 1: Lihat class ID pada gambar — tentukan nama yang benar!', 
             fontsize=13, fontweight='bold', color='red')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR/'verify_classes.png'), dpi=100)
plt.show()

print('LEGEND:')
for i, col in enumerate(COLOR_PALETTE[:len(counts)]):
    print(f'  ID {i} = warna {["ORANGE","CYAN","RED","YELLOW","GREEN","PURPLE","BLUE","PINK"][i]}  <- isi nama di CLASS_NAMES bawah')

## 6. Isi CLASS_NAMES Berdasarkan Visualisasi
**Lihat gambar di Cell 5 lalu isi nama class yang benar di bawah ini.**

In [ ]:
# ============================================================
# WAJIB DIISI: Ganti sesuai visualisasi di atas
# Contoh:
#   0: 'person'     <- ganti dengan nama sebenarnya
#   1: 'no_harness' <- ganti dengan nama sebenarnya
# ============================================================
CLASS_NAMES = {
    0: '[GANTI: class 0]',
    1: '[GANTI: class 1]',
    # tambah sesuai jumlah kelas
}
NC = len(CLASS_NAMES)
# ============================================================

# Validasi
max_id = max(counts) if counts else -1
print(f'NC yang kamu isi  : {NC}')
print(f'Max class ID data : {max_id}')
if max_id >= NC:
    print(f'PERINGATAN: Ada class ID {max_id} tapi CLASS_NAMES hanya sampai {NC-1}!')
    print('           Tambah class yang kurang!')
else:
    print('OK — jumlah class sesuai')

for cid, name in CLASS_NAMES.items():
    if '[GANTI' in name:
        print(f'Belum diisi: class {cid} masih placeholder!')
        break

## 7. Konfirmasi Visualisasi dengan Nama
**Re-visualisasi dengan nama class yang sudah diisi. Pastikan label benar!**

In [ ]:
# Cek tidak ada placeholder
for name in CLASS_NAMES.values():
    if '[GANTI' in str(name):
        raise ValueError('CLASS_NAMES masih ada placeholder! Isi dulu di Cell 6.')

samples2 = random.sample(all_imgs, min(9, len(all_imgs)))
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, ip in zip(axes.flat, samples2):
    img = cv2.imread(str(ip))
    if img is None: ax.axis('off'); continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lp = train_lbl_dir / (ip.stem + '.txt')
    names_found = []
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5: continue
            cid = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = int((xc-bw/2)*w); y1 = int((yc-bh/2)*h)
            x2 = int((xc+bw/2)*w); y2 = int((yc+bh/2)*h)
            col = COLOR_PALETTE[cid % len(COLOR_PALETTE)]
            cv2.rectangle(img, (x1,y1), (x2,y2), col, 2)
            nm = CLASS_NAMES.get(cid, f'cls_{cid}')
            cv2.putText(img, nm, (x1, max(y1-5,15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, col, 2)
            names_found.append(nm)
    ax.imshow(img); ax.axis('off')
    ax.set_title(', '.join(set(names_found)) or '-', fontsize=9)

plt.suptitle('STEP 2: Konfirmasi nama class — apakah sudah benar?',
             fontsize=13, fontweight='bold', color='blue')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR/'confirmed_classes.png'), dpi=100)
plt.show()
print('Jika nama sudah benar, lanjut ke Cell 8 (Buat YAML + Training)')

## 8. Buat data.yaml

In [ ]:
val_split  = 'valid' if (EXTRACT_DIR/'valid'/'images').exists() else 'val'
test_split = 'test'  if (EXTRACT_DIR/'test'/'images').exists()  else val_split

DATA_YAML = OUTPUT_DIR / f'{RUN_NAME}.yaml'
yaml.dump({
    'path' : str(EXTRACT_DIR),
    'train': 'train/images',
    'val'  : f'{val_split}/images',
    'test' : f'{test_split}/images',
    'nc'   : NC,
    'names': CLASS_NAMES,
}, open(DATA_YAML,'w'), default_flow_style=False)
print(f'YAML: {DATA_YAML}')
print(DATA_YAML.read_text())

## 9. Training

In [ ]:
from ultralytics import YOLO

# ============================================================
# Sesuaikan konfigurasi training jika perlu
# ============================================================
EPOCHS   = 100
BATCH    = 16    # kurangi ke 8 jika OOM
IMGSZ    = 640
PATIENCE = 0     # 0 = disable EarlyStopping (train sampai selesai)
FREEZE   = 10    # transfer learning: bekukan backbone
# ============================================================

print(f'epochs={EPOCHS}, batch={BATCH}, imgsz={IMGSZ}, device={DEVICE}')
print(f'EarlyStopping: {"DISABLED" if PATIENCE==0 else f"patience={PATIENCE}"}')

TRAIN_OK = False
try:
    model = YOLO('yolo26n.pt')
    results = model.train(
        data=str(DATA_YAML), epochs=EPOCHS, batch=BATCH, imgsz=IMGSZ,
        device=DEVICE, patience=PATIENCE, freeze=FREEZE, pretrained=True,
        project=str(OUTPUT_DIR/'runs'), name=RUN_NAME, exist_ok=True,
        augment=True, mosaic=1.0, mixup=0.1, copy_paste=0.1,
        hsv_h=0.02, hsv_s=0.75, hsv_v=0.4,
        fliplr=0.5, degrees=8.0, scale=0.5, erasing=0.3,
        optimizer='AdamW', lr0=0.001, lrf=0.01, weight_decay=0.0005, warmup_epochs=3,
        save_period=10, verbose=True, plots=True,
    )
    print('Training selesai!')
    TRAIN_OK = True
except KeyboardInterrupt:
    print('Dihentikan manual — checkpoint:', RUN_DIR/'weights'/'last.pt')
except Exception as e:
    print(f'Error: {e}'); traceback.print_exc()

## 10. Evaluasi

In [ ]:
best_pt = RUN_DIR/'weights'/'best.pt'
last_pt = RUN_DIR/'weights'/'last.pt'
eval_pt = best_pt if best_pt.exists() else last_pt

if not eval_pt.exists():
    print('Tidak ada checkpoint!')
else:
    try:
        if TRAIN_OK:
            m = results.results_dict
            map50=m.get('metrics/mAP50(B)',0); prec=m.get('metrics/precision(B)',0); rec=m.get('metrics/recall(B)',0)
        else:
            em=YOLO(str(eval_pt)); vr=em.val(data=str(DATA_YAML),device=DEVICE,verbose=False)
            map50=vr.box.map50; prec=vr.box.mp; rec=vr.box.mr
        print(f'mAP50     : {map50:.4f}  {"OK" if map50>=0.70 else "Rendah"}')
        print(f'Precision : {prec:.4f}')
        print(f'Recall    : {rec:.4f}')
    except Exception as e:
        print(f'Eval error: {e}')

from IPython.display import Image as IPImg, display
for pn in ['results.png','confusion_matrix.png','PR_curve.png']:
    pf = RUN_DIR/pn
    if pf.exists(): print(f'--- {pn} ---'); display(IPImg(str(pf),width=750))

## 11. Inference Test

In [ ]:
if eval_pt.exists():
    bm = YOLO(str(eval_pt))
    tdir = EXTRACT_DIR/'test'/'images'
    if not tdir.exists(): tdir = EXTRACT_DIR/val_split/'images'
    imgs = list(tdir.glob('*.jpg')) + list(tdir.glob('*.png'))
    if not imgs: print('Tidak ada gambar test')
    else:
        samples3 = random.sample(imgs, min(9, len(imgs)))
        fig,axes=plt.subplots(3,3,figsize=(15,12))
        for ax,ip in zip(axes.flat,samples3):
            try:
                pred=bm.predict(str(ip),conf=0.25,verbose=False)[0]
                ax.imshow(cv2.cvtColor(pred.plot(),cv2.COLOR_BGR2RGB))
                bxs=pred.boxes
                title=', '.join([f"{pred.names[int(b.cls[0])]}:{float(b.conf[0]):.0%}" for b in bxs]) if bxs and len(bxs) else 'no det'
                ax.set_title(title[:40],fontsize=8)
            except Exception as e: ax.set_title(str(e)[:30])
            ax.axis('off')
        plt.suptitle('Inference Test',fontsize=13); plt.tight_layout()
        plt.savefig(str(OUTPUT_DIR/'inference.png'),dpi=100); plt.show()

## 12. Auto-Save Model

In [ ]:
dest = Path('/kaggle/working') if IS_KAGGLE else Path('F:/migas-siews/backend/models')
dest.mkdir(parents=True, exist_ok=True)

best_dst = dest / OUT_NAME
last_dst = dest / OUT_NAME.replace('best_', 'last_')

for src, dst in [(best_pt, best_dst), (last_pt, last_dst)]:
    if src.exists():
        shutil.copy2(src, dst)
        print(f'Saved: {dst}  ({dst.stat().st_size/1e6:.1f} MB)')
    else:
        print(f'Tidak ada: {src}')

if IS_KAGGLE:
    print(f'\nDownload {OUT_NAME} dari tab Output Kaggle')
    print(f'Salin ke: f:/migas-siews/backend/models/{OUT_NAME}')

## Troubleshooting

| Masalah | Solusi |
|---|---|
| `ZIP: None, Exists: False` | Di Kaggle tidak ada zip — dataset otomatis diekstrak |
| Class names terbalik | Lihat Cell 5 (ID angka) dulu sebelum isi CLASS_NAMES |
| mAP rendah | Kurangi `freeze=5`, cek apakah class distribution seimbang |
| OOM | Kurangi `batch=8` atau `imgsz=416` |
| Loss tidak turun | Kurangi `lr0=0.0005`, naikkan `warmup_epochs=5` |